In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
import joblib


# ============================================================
# 1. DEFINE INPUT AND OUTPUT PATHS
# ============================================================

raw_path = "../data_raw/nab/realKnownCause/realKnownCause/machine_temperature_system_failure.csv"

output_dir = Path("../data_processed/NAB")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir)


# ============================================================
# 2. VERIFY THAT THE ORIGINAL DATASET EXISTS
# ============================================================

for root, dirs, files in os.walk("../data_raw"):
    if "machine_temperature_system_failure.csv" in files:
        print("Found:", os.path.join(root, "machine_temperature_system_failure.csv"))


# ============================================================
# 3. LOAD AND SORT THE NAB DATASET
# ============================================================

df = pd.read_csv(raw_path)

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(df.head())
print(df.info())
print(df["value"].describe())
print("Missing values:")
print(df.isna().sum())


# ============================================================
# 4. DEFINE ANOMALY TIME WINDOWS
# ============================================================

anomaly_windows = [
    ("2013-12-10 06:25:00", "2013-12-12 05:35:00"),
    ("2013-12-15 17:50:00", "2013-12-17 17:00:00"),
    ("2014-01-27 14:20:00", "2014-01-29 13:30:00"),
    ("2014-02-07 14:55:00", "2014-02-09 14:05:00"),
]

anomaly_windows = [
    (pd.Timestamp(start), pd.Timestamp(end))
    for start, end in anomaly_windows
]


# ============================================================
# 5. ASSIGN ANOMALY LABELS TO THE DATA
# ============================================================

df["y_true"] = 0

for start, end in anomaly_windows:
    mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
    df.loc[mask, "y_true"] = 1

print("Počet normálnych a anomálnych meraní:")
print(df["y_true"].value_counts())


# ============================================================
# 6. EXTRACT FEATURES FROM THE TIME SERIES
# ============================================================

window = 50

df["diff1"] = df["value"].diff()
df["roll_mean"] = df["value"].rolling(window).mean()
df["roll_std"] = df["value"].rolling(window).std()
df["roll_min"] = df["value"].rolling(window).min()
df["roll_max"] = df["value"].rolling(window).max()

df_features = df.dropna().reset_index(drop=True)

print(df_features.head())
print("Columns:", df_features.columns.tolist())
print("Total rows:", len(df_features))
print("Anomaly count:", df_features["y_true"].sum())


# ============================================================
# 7. SAVE THE LABELED DATASET
# ============================================================

labeled_path = output_dir / "industrial_machine_features_labeled.csv"
df_features.to_csv(labeled_path, index=False)

print("Saved labeled dataset:", labeled_path)


# ============================================================
# 8. SAVE FEATURES ONLY DATASET
# ============================================================

feature_cols = [
    "value",
    "diff1",
    "roll_mean",
    "roll_std",
    "roll_min",
    "roll_max"
]

features_only = df_features[feature_cols].copy()

features_path = output_dir / "industrial_machine_features.csv"
features_only.to_csv(features_path, index=False)

print("Saved feature dataset:", features_path)


# ============================================================
# 9. NORMALIZE FEATURES AND SAVE THE SCALER
# ============================================================

scaler = StandardScaler()
X = scaler.fit_transform(features_only)

scaler_path = output_dir / "nab_scaler.pkl"
joblib.dump(scaler, scaler_path)

print("Saved scaler:", scaler_path)


# ============================================================
# 10. CREATE DATA DISTRIBUTION TABLE
# ============================================================

normal_count = int((df_features["y_true"] == 0).sum())
anomaly_count = int((df_features["y_true"] == 1).sum())
total_count = len(df_features)

split_df = pd.DataFrame({
    "Trieda": ["Normálne merania", "Anomálne merania", "Spolu"],
    "Počet riadkov": [normal_count, anomaly_count, total_count],
    "Podiel (%)": [
        round(normal_count / total_count * 100, 2),
        round(anomaly_count / total_count * 100, 2),
        100.00
    ]
})

print("\nRozdelenie meraní v datasete NAB:")
print(split_df)

split_csv_path = output_dir / "nab_split_table.csv"
split_df.to_csv(split_csv_path, index=False)

fig, ax = plt.subplots(figsize=(7, 2.2))
ax.axis("off")

table = ax.table(
    cellText=split_df.values,
    colLabels=split_df.columns,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#dce6f1")
    else:
        cell.set_facecolor("#f9f9f9")

plt.title("Rozdelenie meraní na normálne a anomálne v datasete NAB", fontsize=13)

split_img_path = output_dir / "nab_split_table.png"
fig.savefig(split_img_path, dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 11. CREATE A PREVIEW TABLE OF THE PROCESSED DATASET
# ============================================================

df_display = df_features.copy()

if "timestamp" in df_display.columns:
    df_display["timestamp"] = pd.to_datetime(df_display["timestamp"]).dt.strftime("%Y-%m-%d")

df_display = df_display.head().round(3)

fig, ax = plt.subplots(figsize=(14, 3))
ax.axis("off")

table = ax.table(
    cellText=df_display.values,
    colLabels=df_display.columns,
    loc="center",
    cellLoc="center"
)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#dce6f1")
    else:
        cell.set_facecolor("#f9f9f9")

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

plt.title(
    "Ukážka extrahovaných príznakov a označení anomálií z datasetu NAB",
    fontsize=14
)

table_img_path = output_dir / "industrial_table_labeled.png"
fig.savefig(table_img_path, dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 12. GENERATE FEATURE STATISTICS TABLE
# ============================================================

stats_df = df_features[feature_cols].describe().round(3)
stats_df = stats_df.reset_index().rename(columns={"index": "Štatistika"})

print("\nZákladné štatistiky príznakov:")
print(stats_df)

stats_csv_path = output_dir / "nab_feature_stats_table.csv"
stats_df.to_csv(stats_csv_path, index=False)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis("off")

table = ax.table(
    cellText=stats_df.values,
    colLabels=stats_df.columns,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#dce6f1")
    else:
        cell.set_facecolor("#f9f9f9")

plt.title("Základné štatistiky extrahovaných príznakov datasetu NAB", fontsize=13)

stats_img_path = output_dir / "nab_feature_stats_table.png"
fig.savefig(stats_img_path, dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 13. VISUALIZE A NORMAL TIME SERIES SEGMENT
# ============================================================

normal_df = df_features[df_features["y_true"] == 0].copy()

plt.figure(figsize=(12, 4))
plt.plot(range(len(normal_df.head(300))), normal_df["value"].head(300).values)
plt.title("Príklad normálneho úseku časového radu")
plt.xlabel("Index")
plt.ylabel("Hodnota")
plt.grid(True)
plt.tight_layout()

normal_img_path = output_dir / "nab_normal_segment.png"
plt.savefig(normal_img_path, dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 14. VISUALIZE AN ANOMALOUS TIME SERIES SEGMENT
# ============================================================

anomaly_df = df_features[df_features["y_true"] == 1].copy()

plt.figure(figsize=(12, 4))
plt.plot(range(len(anomaly_df.head(300))), anomaly_df["value"].head(300).values)
plt.title("Príklad anomálneho úseku časového radu")
plt.xlabel("Index")
plt.ylabel("Hodnota")
plt.grid(True)
plt.tight_layout()

anomaly_img_path = output_dir / "nab_anomaly_segment.png"
plt.savefig(anomaly_img_path, dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 15. VISUALIZE THE COMPLETE SIGNAL WITH ANOMALIES
# ============================================================

plt.figure(figsize=(15, 5))
plt.plot(df_features["value"].values, label="Hodnota")

plt.scatter(
    df_features.index[df_features["y_true"] == 1],
    df_features.loc[df_features["y_true"] == 1, "value"],
    label="Anomálie",
    marker="o"
)

plt.title("Časový rad NAB s označenými anomáliami")
plt.xlabel("Index")
plt.ylabel("Hodnota")
plt.legend()
plt.grid(True)
plt.tight_layout()

full_signal_img_path = output_dir / "nab_full_signal_anomalies.png"
plt.savefig(full_signal_img_path, dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 16. DISPLAY ALL SAVED FILES
# ============================================================

print("\nFiles saved to:", output_dir)
for file in sorted(output_dir.iterdir()):
    print("-", file.name)